# Policy Transformer Training (vocab1 / 1793) — production warm-start

Self-contained Kaggle notebook. Warm-starts from production **`ultra_3o9.pt`** (vocab 4033)
by remapping emb/`lm_head` rows onto **`vocab1.pkl`** (1793), then fine-tunes ~30k steps
so the policy keeps production strength without the large-vocab overhead.

Production checkpoints are **read-only** (never modified).

## Exact upload steps

### 1) Create a Kaggle Dataset

Upload these files:

| Local path | Upload as |
|---|---|
| `data/full_X.pt` | `full_X.pt` |
| `data/full_Y.pt` | `full_Y.pt` |
| `data/weak_X.pt` | `weak_X.pt` |
| `data/weak_Y.pt` | `weak_Y.pt` |
| `data/elite_X.pt` | `elite_X.pt` |
| `data/elite_Y.pt` | `elite_Y.pt` |
| `policy_transformer/vocab1.pkl` | `vocab1.pkl` |
| `policy_transformer/vocab.pkl` | `vocab.pkl` |
| `policy_transformer/ultra_3o9.pt` | `ultra_3o9.pt` |

Optional: a previous `ultra_v1_o1.pt` / `best_ultra_v1_o1.pt` to **resume** (skips production remap).

### 2) Kaggle notebook settings

- Accelerator: **GPU T4** (or better)
- Add Data → attach the dataset above
- Run all cells top → bottom
- Download `best_ultra_v1_o1.pt` (prefer) and/or `ultra_v1_o1.pt` from `/kaggle/working/`

### 3) Sanity

Step 0 CE after warm-start should be **~1.9–2.5** (not ~7.5). The notebook asserts CE < 4.0.


In [ ]:
%%writefile model.py
import torch
import torch.nn as nn
from torch.nn import functional as F
import math


def _apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    Apply rotary position embedding to a tensor shaped [B, n_head, T, head_dim].
    cos/sin are shaped [T, head_dim//2] (broadcasted inside).
    """
    # x: [..., head_dim] where head_dim is even
    x_even = x[..., ::2]
    x_odd = x[..., 1::2]
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    out_even = x_even * cos - x_odd * sin
    out_odd = x_even * sin + x_odd * cos
    return torch.stack((out_even, out_odd), dim=-1).flatten(-2)

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        # output projection
        self.c_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout
        head_dim = n_embd // n_head
        assert head_dim % 2 == 0, "RoPE requires an even head dimension"

        # RoPE (rotary position embeddings) cache (non-persistent so old checkpoints still load)
        inv_freq = 1.0 / (10000.0 ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        self.register_buffer("_rope_inv_freq", inv_freq, persistent=False)
        self.register_buffer("_rope_cos_cached", torch.empty(0), persistent=False)
        self.register_buffer("_rope_sin_cached", torch.empty(0), persistent=False)
        self._rope_seq_len_cached = 0
        self._rope_cache_dtype = None
        self._rope_cache_device = None

        # Flash Attention requires PyTorch 2.0+
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: Flash Attention not available. Using slow manual attention.")
            self.register_buffer("bias", torch.tril(torch.ones(block_size, block_size))
                                        .view(1, 1, block_size, block_size))

    def _rope_cos_sin(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> tuple[torch.Tensor, torch.Tensor]:
        # Cache per (device, dtype, seq_len>=T). Compute in fp32 for stability, cast for use.
        if (
            self._rope_seq_len_cached >= seq_len
            and self._rope_cache_device == device
            and self._rope_cache_dtype == dtype
            and self._rope_cos_cached.numel() != 0
        ):
            return self._rope_cos_cached[:seq_len], self._rope_sin_cached[:seq_len]

        t = torch.arange(seq_len, device=device, dtype=torch.float32)
        freqs = torch.outer(t, self._rope_inv_freq.to(device=device))  # [T, head_dim//2]
        cos = freqs.cos().to(dtype=dtype)
        sin = freqs.sin().to(dtype=dtype)

        self._rope_cos_cached = cos
        self._rope_sin_cached = sin
        self._rope_seq_len_cached = seq_len
        self._rope_cache_device = device
        self._rope_cache_dtype = dtype
        return cos, sin

    def forward(self, x):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        # RoPE: apply rotary embeddings to q/k (not v)
        cos, sin = self._rope_cos_sin(T, device=q.device, dtype=q.dtype)
        q = _apply_rope(q, cos, sin)
        k = _apply_rope(k, cos, sin)

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(
                q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True
            )
        else:
            # manual implementation (fallback)
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side
        y = self.resid_dropout(self.c_proj(y))
        return y

class FeedFoward(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.sa = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedFoward(n_embd, dropout)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class PolicyHead(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer, dropout, device):
        super().__init__()
        self.block_size = block_size
        self.device = device
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        # Ensure T is within block_size
        if T > self.block_size:
            idx = idx[:, -self.block_size:]
            T = self.block_size
            if targets is not None:
                targets = targets[:, -self.block_size:]

        tok_emb = self.token_embedding_table(idx) 
        # With RoPE applied inside attention, we don't add absolute positional embeddings here.
        x = tok_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        if targets is None:
            logits = self.lm_head(x[:, [-1], :]) # optimization: only compute last logit for inference
            loss = None
        else:
            logits = self.lm_head(x)
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets, ignore_index=-100) # Added ignore_index explicitly
        
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


In [ ]:
# ───────────────────────────────────────────────────────────────
# KAGGLE DATA SETUP: Auto-find + symlink required training files
# ───────────────────────────────────────────────────────────────
import os

input_root = "/kaggle/input"

# Required for training + production warm-start.
REQUIRED_FILES = {
    "vocab1.pkl": "vocab1.pkl",
    "vocab.pkl": "vocab.pkl",
    "ultra_3o9.pt": "ultra_3o9.pt",
    "full_X.pt": "full_X.pt",
    "full_Y.pt": "full_Y.pt",
    "elite_X.pt": "elite_X.pt",
    "elite_Y.pt": "elite_Y.pt",
    "weak_X.pt": "weak_X.pt",
    "weak_Y.pt": "weak_Y.pt",
}

# Optional: set to a filename you uploaded if resuming a vocab1 run, else None.
# If set, production remapping is skipped and this checkpoint is loaded instead.
RESUME_CHECKPOINT = None  # e.g. "best_ultra_v1_o1.pt"

OPTIONAL_FILES = {}
if RESUME_CHECKPOINT:
    OPTIONAL_FILES[RESUME_CHECKPOINT] = RESUME_CHECKPOINT

files_to_link = {**REQUIRED_FILES, **OPTIONAL_FILES}
found = {local: False for local in files_to_link.values()}

for root, dirs, files in os.walk(input_root):
    for original_name, local_name in files_to_link.items():
        if original_name in files and not found[local_name]:
            src = os.path.join(root, original_name)
            dst = local_name
            print(f"Found {original_name} in: {src}")
            if os.path.lexists(dst):
                os.remove(dst)
            os.symlink(src, dst)
            found[local_name] = True
            print(f"  → Symlinked to ./{dst}")

print("\nSummary:")
missing_required = []
for local in REQUIRED_FILES.values():
    ok = found.get(local, False)
    print(f"  {local}: {'OK Found & linked' if ok else 'MISSING'}")
    if not ok:
        missing_required.append(local)

for local in OPTIONAL_FILES.values():
    ok = found.get(local, False)
    print(f"  {local} (optional): {'OK Found & linked' if ok else 'skip (fresh warm-start)'}")

if missing_required:
    raise FileNotFoundError(
        "Missing required files: "
        + ", ".join(missing_required)
        + ". Add your dataset via 'Add Data' and re-run this cell."
    )

print("\nAll required files linked.")


## Train

**Kaggle T4 usable VRAM is ~14.56 GiB, not 16.** Keep `USE_TORCH_COMPILE=False` and `B=320`.
If you OOM after a failed run, **Restart session** first (fragmented VRAM), then re-run.
Re-run this cell to continue; set `RESUME_CHECKPOINT` in the link cell first.


In [ ]:
# ──────────────────────────────────────────────────────────────
# Train policy transformer on vocab1 (1793)
# ──────────────────────────────────────────────────────────────
import math
import os
import pickle
import gc

import torch
from torch.amp import GradScaler, autocast
from tqdm import tqdm

from model import PolicyHead

# ── Hyperparameters (matches ultra_3o9 architecture; vocab1 size) ──
grad_accum = 1
block_size = 128
max_iters = 100_001
run_iters = 30_000          # how many steps THIS session will run
eval_interval = 1_000
save_interval = 1_000
learning_rate = 1e-4        # fine-tune LR after production warm-start (not scratch)
min_lr = learning_rate / 10
warmup_iters = 1_000
lr_decay_iters = 30_000
n_embd = 500
n_head = 10
n_layer = 8
dropout = 0.1
eval_iters = 50
weight_decay = 1e-4

# Batch mixture (total batch = n_normal + n_elite + n_weak)
n_normal = 256
n_elite = 32
n_weak = 32
batch_size = n_normal + n_elite + n_weak  # 320 — fits T4 14.56GiB + backward

# Output checkpoint written under /kaggle/working
output_checkpoint = "ultra_v1_o1.pt"
# Resume path: only used if RESUME_CHECKPOINT was set & linked in the previous cell
checkpoint_path = RESUME_CHECKPOINT if (RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT)) else None

USE_TORCH_COMPILE = False  # compile OOMs backward on Kaggle T4; leave off

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "Enable a GPU accelerator in the Kaggle notebook settings."
scaler = GradScaler(enabled=True)

# ── Vocab (MUST be vocab1 / 1793) ──
with open("vocab1.pkl", "rb") as f:
    stoi = pickle.load(f)
itos = {i: s for s, i in stoi.items()}
vocab_size = len(stoi)
assert "|" in stoi, "vocab1.pkl missing pad token '|'"
assert vocab_size == 1793, f"Expected vocab1 size 1793, got {vocab_size}. Did you upload vocab.pkl by mistake?"
print(f"Vocab size: {vocab_size}")

# ── Data loading ──
def load_and_split(x_file: str, y_file: str, train_ratio: float = 0.9):
    print(f"Loading {x_file} / {y_file} ...")
    full_X = torch.load(x_file, map_location="cpu", weights_only=False)
    full_Y = torch.load(y_file, map_location="cpu", weights_only=False)
    gc.collect()

    assert full_X.shape == full_Y.shape, f"Shape mismatch: {full_X.shape} vs {full_Y.shape}"
    print("Corpus shapes:", tuple(full_X.shape), tuple(full_Y.shape))
    print("X min/max:", int(full_X.min()), int(full_X.max()))
    print("Y min/max:", int(full_Y.min()), int(full_Y.max()))

    assert int(full_X.min()) >= 0
    assert int(full_X.max()) < vocab_size, (
        f"X max token {int(full_X.max())} >= vocab_size {vocab_size}. "
        "Data was not tokenized with this vocab1.pkl."
    )
    # Y uses -100 as ignore_index for masked positions
    y_pos = full_Y[full_Y >= 0]
    if len(y_pos):
        assert int(y_pos.max()) < vocab_size, (
            f"Y max token {int(y_pos.max())} >= vocab_size {vocab_size}."
        )

    split_idx = int(train_ratio * len(full_X))
    X_train, Y_train = full_X[:split_idx], full_Y[:split_idx]
    X_val, Y_val = full_X[split_idx:], full_Y[split_idx:]

    del full_X, full_Y
    gc.collect()
    print(f"Train tokens: {len(X_train):,}, Val tokens: {len(X_val):,}")
    return X_train, Y_train, X_val, Y_val


X_train, Y_train, X_val, Y_val = load_and_split("full_X.pt", "full_Y.pt")
# IMPORTANT: do NOT swap elite/weak (old notebook had this bug)
X_train_weak, Y_train_weak, X_val_weak, Y_val_weak = load_and_split("weak_X.pt", "weak_Y.pt")
X_train_elite, Y_train_elite, X_val_elite, Y_val_elite = load_and_split("elite_X.pt", "elite_Y.pt")


def get_batch(split: str):
    if split == "train":
        data_x, data_y = X_train, Y_train
        elite_x, elite_y = X_train_elite, Y_train_elite
        weak_x, weak_y = X_train_weak, Y_train_weak
    else:
        data_x, data_y = X_val, Y_val
        elite_x, elite_y = X_val_elite, Y_val_elite
        weak_x, weak_y = X_val_weak, Y_val_weak

    # Guard tiny val splits
    def _rand_starts(n_src, n_batch):
        upper = max(1, len(n_src) - block_size + 1)
        return torch.randint(0, upper, (n_batch,))

    ix = _rand_starts(data_x, n_normal)
    ixe = _rand_starts(elite_x, n_elite)
    ixw = _rand_starts(weak_x, n_weak)

    xb = torch.stack([data_x[i : i + block_size] for i in ix])
    yb = torch.stack([data_y[i : i + block_size] for i in ix])
    xbe = torch.stack([elite_x[i : i + block_size] for i in ixe])
    ybe = torch.stack([elite_y[i : i + block_size] for i in ixe])
    xbw = torch.stack([weak_x[i : i + block_size] for i in ixw])
    ybw = torch.stack([weak_y[i : i + block_size] for i in ixw])

    x = torch.cat([xb, xbe, xbw], dim=0).to(device, dtype=torch.long, non_blocking=True)
    y = torch.cat([yb, ybe, ybw], dim=0).to(device, dtype=torch.long, non_blocking=True)
    return x, y


@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters, device=device)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            with autocast(device_type="cuda"):
                _, loss = model(xb, yb)
            losses[k] = loss
        out[split] = losses.mean().item()
    model.train()
    return out


def get_lr(it: int) -> float:
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    if it > lr_decay_iters:
        return min_lr
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)


def raw_state_dict(model: torch.nn.Module) -> dict:
    """Strip torch.compile wrappers so checkpoints load cleanly in inference.py."""
    sd = model.state_dict()
    if any(k.startswith("_orig_mod.") for k in sd):
        sd = {k.replace("_orig_mod.", "", 1): v for k, v in sd.items()}
    return sd


def save_checkpoint(model, optimizer, scaler, iter_num, best_val_loss, path: str):
    checkpoint = {
        "model": raw_state_dict(model),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "iter_num": iter_num,
        "best_val_loss": best_val_loss,
        "config": {
            "vocab_size": vocab_size,
            "n_embd": n_embd,
            "n_head": n_head,
            "n_layer": n_layer,
            "block_size": block_size,
            "dropout": dropout,
        },
    }
    torch.save(checkpoint, path)
    print(f"Saved checkpoint → {path} (iter {iter_num})")


def main():
    torch.set_float32_matmul_precision("high")

    model = PolicyHead(vocab_size, n_embd, block_size, n_head, n_layer, dropout, device)
    model = model.to(device)
    warm_started = False

    def build_vocab1_policy_state(legacy_ckpt_path, vocab_old_path, vocab_new_path):
        """Remap production PolicyHead (4033) → vocab1 (1793) by UCI string."""
        with open(vocab_old_path, "rb") as f:
            stoi_old = pickle.load(f)
        with open(vocab_new_path, "rb") as f:
            stoi_new = pickle.load(f)
        missing = [t for t in stoi_new if t not in stoi_old]
        if missing:
            raise ValueError(f"{len(missing)} vocab1 tokens missing from production vocab")
        ckpt = torch.load(legacy_ckpt_path, map_location="cpu", weights_only=False)
        state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
        if hasattr(state, "state_dict"):
            state = state.state_dict()
        state = {k.removeprefix("_orig_mod."): v for k, v in state.items()}
        emb_old = state["token_embedding_table.weight"]
        assert emb_old.shape[0] == len(stoi_old), (
            f"ckpt vocab rows {emb_old.shape[0]} != len(vocab.pkl) {len(stoi_old)}"
        )
        itos_new = {int(i): s for s, i in stoi_new.items()}
        gather = torch.tensor([int(stoi_old[itos_new[i]]) for i in range(len(stoi_new))], dtype=torch.long)
        out = {k: v for k, v in state.items() if k not in (
            "token_embedding_table.weight", "lm_head.weight", "lm_head.bias"
        )}
        out["token_embedding_table.weight"] = emb_old.index_select(0, gather).contiguous()
        out["lm_head.weight"] = state["lm_head.weight"].index_select(0, gather).contiguous()
        out["lm_head.bias"] = state["lm_head.bias"].index_select(0, gather).contiguous()
        return out

    warm_started = False
    if not checkpoint_path:
        print("Warm-starting from production ultra_3o9.pt → vocab1 remap ...")
        remapped = build_vocab1_policy_state("ultra_3o9.pt", "vocab.pkl", "vocab1.pkl")
        # Drop unused absolute pos emb if present; PolicyHead still has the table
        model.load_state_dict(remapped, strict=False)
        warm_started = True
        print(f"Loaded remapped production weights ({len(remapped)} tensors)")

    if USE_TORCH_COMPILE:
        try:
            model = torch.compile(model, mode="default")
            print("torch.compile enabled")
        except Exception as e:
            print(f"torch.compile failed ({e}); continuing without it")

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    iter_num = 0
    best_val_loss = 1e9
    if checkpoint_path:
        print(f"Attempting resume from {checkpoint_path} ...")
        try:
            ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
            state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
            if isinstance(state, dict) and any(k.startswith("_orig_mod.") for k in state):
                state = {k.replace("_orig_mod.", "", 1): v for k, v in state.items()}

            # Fail fast on vocab mismatch (e.g. old 4033 checkpoint)
            emb_key = "token_embedding_table.weight"
            if emb_key in state and state[emb_key].shape[0] != vocab_size:
                raise RuntimeError(
                    f"Checkpoint vocab {state[emb_key].shape[0]} != current {vocab_size}. "
                    "Start fresh (set RESUME_CHECKPOINT=None) or use a vocab1 checkpoint."
                )

            # Load into potentially-compiled model
            target = model._orig_mod if hasattr(model, "_orig_mod") else model
            target.load_state_dict(state)
            if isinstance(ckpt, dict):
                if "optimizer" in ckpt:
                    optimizer.load_state_dict(ckpt["optimizer"])
                if "scaler" in ckpt:
                    scaler.load_state_dict(ckpt["scaler"])
                iter_num = int(ckpt.get("iter_num", 0))
                best_val_loss = float(ckpt.get("best_val_loss", 1e9))
            print(f"Resumed at iter {iter_num}, best_val_loss={best_val_loss:.4f}")
        except Exception as e:
            print(f"Checkpoint load failed: {e}. Starting fresh.")
            iter_num = 0
            best_val_loss = 1e9

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"Device: {device}")
    print(f"Parameters: {n_params:.2f}M")
    print(f"Batch size: {batch_size} (normal={n_normal}, elite={n_elite}, weak={n_weak})")
    print(f"Effective batch size: {batch_size * grad_accum}")
    print(f"Will run iters [{iter_num} → {min(max_iters, iter_num + run_iters)})")

    model.train()
    target_end_iter = min(max_iters, iter_num + run_iters)
    accum_loss = 0.0
    
    # Step-0 CE sanity: warm-start should be near production (~2), not random (~7.5)
    model.eval()
    with torch.no_grad():
        xb0, yb0 = get_batch("train")
        with autocast(device_type="cuda"):
            _, loss0 = model(xb0, yb0)
        ce0 = float(loss0)
    model.train()
    print(f"Step-0 train CE (1 batch): {ce0:.4f}")
    if warm_started or not checkpoint_path:
        assert ce0 < 4.0, (
            f"Warm-start failed: step-0 CE={ce0:.4f} (expected < 4). "
            "Check ultra_3o9.pt + vocab.pkl remap."
        )

    pbar = tqdm(total=target_end_iter - iter_num, desc="Training", unit="iter", dynamic_ncols=True)

    while iter_num < target_end_iter:
        lr = get_lr(iter_num)
        for g in optimizer.param_groups:
            g["lr"] = lr

        if iter_num > 0 and (iter_num % eval_interval == 0):
            losses = estimate_loss(model)
            print(
                f"step {iter_num}: train loss {losses['train']:.4f}, "
                f"val loss {losses['val']:.4f}, lr {lr:.2e}"
            )
            if losses["val"] < best_val_loss:
                best_val_loss = losses["val"]
                save_checkpoint(model, optimizer, scaler, iter_num, best_val_loss, f"best_{output_checkpoint}")

        optimizer.zero_grad(set_to_none=True)
        accum_loss = 0.0
        for _ in range(grad_accum):
            xb, yb = get_batch("train")
            with autocast(device_type="cuda"):
                _, loss = model(xb, yb)
                loss = loss / grad_accum
            scaler.scale(loss).backward()
            accum_loss += loss.item()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        if iter_num % 50 == 0:
            print(f"step {iter_num}: train loss {accum_loss:.4f}, lr {lr:.2e}")

        iter_num += 1
        pbar.update(1)

        if iter_num % save_interval == 0:
            save_checkpoint(model, optimizer, scaler, iter_num, best_val_loss, output_checkpoint)

    pbar.close()
    save_checkpoint(model, optimizer, scaler, iter_num, best_val_loss, output_checkpoint)
    losses = estimate_loss(model)
    print(f"step {iter_num}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    print(f"Finished at iter {iter_num}. Download `{output_checkpoint}` from /kaggle/working.")


main()
